In [1]:
import xarray as xr
import xesmf as xe
import os

cesm_to_era5_varnames = {
    "sst": "sea_surface_temperature",
    "icefrac": "sea_ice_cover",
    "psl": "mean_sea_level_pressure",
    "geopotential": "geopotential_500hPa",
    "t2m": "2m_temperature",
}

cesm_to_era5_short_varnames = {
    "sst": "sst",
    "icefrac": "siconc",
    "psl": "msl",
    "geopotential": "z",
    "t2m": "t2m"
}

def transform_obs_to_cesm_format(var_name, output_grid, savedir): 
    os.makedirs(savedir, exist_ok=True)
    savepath = os.path.join(savedir, f"{var_name}_obs_2425.nc")
    if (os.path.exists(savepath)): return 

    ds = xr.open_dataset(os.path.join("/oak/stanford/groups/earlew/yuchen", f"ERA5/{cesm_to_era5_varnames[var_name]}_2425.nc")).sel(latitude=slice(-30,-90))
    weight_file = '/oak/stanford/groups/earlew/yuchen/cesm_lens/grids/era5_small_to_sps_bilinear_regridding_weights.nc'
    
    if os.path.exists(weight_file):
        regridder = xe.Regridder(ds, output_grid, 'bilinear', weights=weight_file, 
                                ignore_degenerate=True, reuse_weights=True, periodic=True)
    else:
        regridder = xe.Regridder(ds, output_grid, 'bilinear', filename=weight_file, 
                                ignore_degenerate=True, reuse_weights=False, periodic=True)
    ds_regridded = regridder(ds)
    ds_regridded = ds_regridded.sel(expver=1).combine_first(ds_regridded.sel(expver=5))

    # rename variable
    ds_regridded = ds_regridded.rename({cesm_to_era5_short_varnames[var_name]: var_name})

    # save
    ds_regridded.to_netcdf(savepath)